In [1]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Aryan@2010",  # ← your password
    database="movie_recommender"
)
cursor = conn.cursor()

import warnings
warnings.filterwarnings('ignore')

movies  = pd.read_sql("SELECT * FROM silver_movies", conn)
ratings = pd.read_sql("SELECT * FROM silver_ratings", conn)

print("✅ Loaded from Silver!")
print("Movies:", movies.shape, "| Ratings:", ratings.shape)

✅ Loaded from Silver!
Movies: (9708, 5) | Ratings: (100836, 4)


In [2]:
# Average rating and total ratings per movie
movie_stats = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

# Round avg rating to 2 decimal places
movie_stats['avg_rating'] = movie_stats['avg_rating'].round(2)

# Merge with movie info
gold_movies = movies.merge(movie_stats, on='movieId', how='left')
gold_movies['avg_rating'] = gold_movies['avg_rating'].fillna(0)
gold_movies['num_ratings'] = gold_movies['num_ratings'].fillna(0)

print("✅ Movie stats created!")
print(gold_movies[['clean_title', 'avg_rating', 'num_ratings']].head(5))

✅ Movie stats created!
                   clean_title  avg_rating  num_ratings
0                    Toy Story        3.92        215.0
1                      Jumanji        3.43        110.0
2             Grumpier Old Men        3.26         52.0
3            Waiting to Exhale        2.36          7.0
4  Father of the Bride Part II        3.07         49.0


In [3]:
# Split genres into separate columns (one-hot encoding)
genres_dummies = movies['genres'].str.get_dummies(sep=', ')

# Combine with movie info
gold_genre_matrix = pd.concat([movies[['movieId', 'clean_title']], genres_dummies], axis=1)

print("✅ Genre matrix created!")
print("Shape:", gold_genre_matrix.shape)
print("Genres found:", list(genres_dummies.columns))

✅ Genre matrix created!
Shape: (9708, 21)
Genres found: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [4]:
# Save gold_movies
cursor.execute("DROP TABLE IF EXISTS gold_movies")
cursor.execute("""
    CREATE TABLE gold_movies (
        movieId INT,
        title VARCHAR(255),
        genres VARCHAR(500),
        clean_title VARCHAR(255),
        year INT,
        avg_rating FLOAT,
        num_ratings INT
    )
""")

for _, row in gold_movies.iterrows():
    try:
        year = int(row['year']) if pd.notna(row['year']) else None
    except:
        year = None
    cursor.execute(
        "INSERT INTO gold_movies VALUES (%s, %s, %s, %s, %s, %s, %s)",
        (int(row['movieId']), row['title'], row['genres'],
         row['clean_title'], year,
         float(row['avg_rating']), int(row['num_ratings']))
    )

conn.commit()
print("✅ Gold layer saved to MySQL!")

✅ Gold layer saved to MySQL!


In [5]:
# Save genre matrix as CSV for the ML step
gold_genre_matrix.to_csv('../data/final/genre_matrix.csv', index=False)
gold_movies.to_csv('../data/final/gold_movies.csv', index=False)

print("✅ Gold CSVs saved to data/final/")

✅ Gold CSVs saved to data/final/


In [6]:
cursor.execute("SELECT COUNT(*) FROM gold_movies")
print("Movies in Gold:", cursor.fetchone()[0])

# Top 5 highest rated movies (with at least 50 ratings)
top_rated = gold_movies[gold_movies['num_ratings'] >= 50].sort_values(
    'avg_rating', ascending=False
).head(5)

print("\n🏆 Top 5 Highest Rated Movies:")
print(top_rated[['clean_title', 'avg_rating', 'num_ratings']].to_string(index=False))

Movies in Gold: 9708

🏆 Top 5 Highest Rated Movies:
                                                         clean_title  avg_rating  num_ratings
                                           Shawshank Redemption, The        4.43        317.0
                                                      Godfather, The        4.29        192.0
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb        4.27         97.0
                                                      Cool Hand Luke        4.27         57.0
                                                          Fight Club        4.27        218.0
